# 09 - PyTorch Dataset and Transforms

**Pipeline:** Skin Cancer 3-Class Classification (NV / MEL / BCC)  
**Purpose:** Read final preprocessed manifests from Section 08, validate all image paths, define PyTorch transforms, implement `SkinLesionDataset`, create DataLoaders for both 3-class and binary cancer-risk tasks, and validate batches.

**Rules:** No training. No model evaluation. No new splits. No image modification. No augmentation saved to disk. No preprocessing/cropping repeated here.

---

## Section 0 - Imports and Environment Check

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

# Environment
CUDA_AVAILABLE = torch.cuda.is_available()
try:
    DEVICE         = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
    GPU_NAME       = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "N/A"
except AssertionError:
    DEVICE = torch.device("cpu")
    GPU_NAME = "N/A"
    CUDA_AVAILABLE = False

print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"CUDA        : {CUDA_AVAILABLE}")
print(f"GPU         : {GPU_NAME}")
print(f"Device      : {DEVICE}")

torch       : 2.6.0+cu124
torchvision : 0.21.0+cu124
CUDA        : True
GPU         : NVIDIA GeForce RTX 4060 Ti
Device      : cuda


## Section 1 - Paths and Configuration

In [2]:
OUTPUT_ROOT        = Path(r"C:\SKIN CANCER v2\pipe output")
FINAL_DATASET_ROOT = Path(r"C:\SKIN CANCER v2\final DS")
PREPROC_DIR        = OUTPUT_ROOT / "preprocessing"
PYTORCH_DIR        = OUTPUT_ROOT / "pytorch_dataset"

CLASS_NAMES  = ["NV", "MEL", "BCC"]
CLASS_INDEX  = {"NV": 0, "MEL": 1, "BCC": 2}
BINARY_INDEX = {"NV": 0, "MEL": 1, "BCC": 1}   # 0=non_cancer, 1=cancer_risk
BINARY_NAMES = ["non_cancer", "cancer_risk"]

# Expected counts from Section 07 freeze (informational — no hard raises)
EXPECTED_COUNTS = {
    "train": {"total": 14332, "NV": 8928, "MEL": 3144, "BCC": 2260},
    "val":   {"total":  3012, "NV": 1886, "MEL":  647, "BCC":  479},
    "test":  {"total":  3045, "NV": 1894, "MEL":  639, "BCC":  512},
}
EXPECTED_TOTAL = 20389

# Transform / DataLoader settings
IMG_SIZE    = 224
RESIZE_TO   = 256
BATCH_SIZE  = 32
NUM_WORKERS = 0       # 0 = safe on Windows with multiprocessing
PIN_MEMORY  = CUDA_AVAILABLE

# ImageNet normalisation constants
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Required columns in preprocessed manifests
REQUIRED_COLS = [
    "preprocessed_full_path", "final_authoritative_label", "class_index",
    "original_full_path", "vignette_crop_applied", "output_image_exists",
]

print("Configuration loaded.")
print(f"  IMG_SIZE={IMG_SIZE}  RESIZE_TO={RESIZE_TO}  BATCH_SIZE={BATCH_SIZE}")
print(f"  NUM_WORKERS={NUM_WORKERS}  PIN_MEMORY={PIN_MEMORY}")

Configuration loaded.
  IMG_SIZE=224  RESIZE_TO=256  BATCH_SIZE=32
  NUM_WORKERS=0  PIN_MEMORY=True


## Section 2 - Create Output Folder

In [3]:
PYTORCH_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {PYTORCH_DIR}")

Ready: C:\SKIN CANCER v2\pipe output\pytorch_dataset


## Section 3 - Load Preprocessed Manifests

In [4]:
manifest_paths = {
    "train": PREPROC_DIR / "train_manifest_preprocessed.csv",
    "val":   PREPROC_DIR / "val_manifest_preprocessed.csv",
    "test":  PREPROC_DIR / "test_manifest_preprocessed.csv",
}

dfs = {}
for split, p in manifest_paths.items():
    if not p.exists():
        raise FileNotFoundError(
            f"Preprocessed manifest not found: {p}\n"
            "Re-run Section 08 first."
        )
    dfs[split] = pd.read_csv(p, low_memory=False)
    print(f"Loaded {split}: {len(dfs[split]):,} rows")

total_loaded = sum(len(v) for v in dfs.values())
print(f"\nTotal rows loaded: {total_loaded:,}")

# Ensure split_assignment column is present
# Section 08 manifests may have 'split_assignment' or 'split'
split_col = {}
for split, df in dfs.items():
    if "split_assignment" in df.columns:
        split_col[split] = "split_assignment"
    elif "split" in df.columns:
        split_col[split] = "split"
    else:
        df["split_assignment"] = split
        split_col[split] = "split_assignment"
        print(f"  WARNING: no split column in {split} — added synthetic column")

Loaded train: 14,332 rows
Loaded val: 3,012 rows
Loaded test: 3,045 rows

Total rows loaded: 20,389


In [5]:
# Repair class_index if missing or unreliable
for split, df in dfs.items():
    if "class_index" not in df.columns or df["class_index"].isna().any():
        df["class_index"] = df["final_authoritative_label"].map(CLASS_INDEX)
        print(f"  {split}: class_index regenerated from final_authoritative_label")

# Add binary_label column
for split, df in dfs.items():
    df["binary_label"] = df["final_authoritative_label"].map(BINARY_INDEX)

# Validate required columns
print("\nRequired column check:")
for split, df in dfs.items():
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        print(f"  {split}: MISSING {missing}")
    else:
        print(f"  {split}: all required columns present")


Required column check:
  train: all required columns present
  val: all required columns present
  test: all required columns present


## Section 4 - Validate Manifest Integrity

In [6]:
integrity = {}
print("=== MANIFEST INTEGRITY CHECK ===")

for split, df in dfs.items():
    n        = len(df)
    exp_tot  = EXPECTED_COUNTS[split]["total"]
    cc       = df["final_authoritative_label"].value_counts()
    dup_mask = df["preprocessed_full_path"].duplicated()
    n_dup    = int(dup_mask.sum())

    # File existence check
    if "output_image_exists" in df.columns:
        n_missing = int((df["output_image_exists"] == False).sum())
    else:
        n_missing = int(df["preprocessed_full_path"]
                        .apply(lambda p: not os.path.exists(str(p))).sum())

    # Readability check (sample up to 50 rows for speed)
    sample_paths = df["preprocessed_full_path"].dropna().head(50)
    n_unreadable = 0
    for p in sample_paths:
        try:
            with Image.open(str(p)) as img:
                img.verify()
        except Exception:
            n_unreadable += 1

    # Label validity
    bad_labels = set(df["final_authoritative_label"].unique()) - set(CLASS_NAMES)

    integrity[split] = {
        "row_count":    n, "expected_rows": exp_tot,
        "n_missing":    n_missing, "n_unreadable_sample": n_unreadable,
        "n_duplicates": n_dup, "bad_labels": bad_labels,
    }

    row_ok  = "OK" if n == exp_tot else f"DIFF exp={exp_tot:,}"
    file_ok = "OK" if n_missing == 0 else f"WARN {n_missing} missing"
    read_ok = "OK" if n_unreadable == 0 else f"WARN {n_unreadable}/50 unreadable"
    dup_ok  = "OK" if n_dup == 0 else f"WARN {n_dup} dups"
    lbl_ok  = "OK" if not bad_labels else f"BAD {bad_labels}"

    print(f"  {split}:")
    print(f"    rows      : {n:,}  {row_ok}")
    print(f"    files     : {file_ok}")
    print(f"    readability (sample 50): {read_ok}")
    print(f"    duplicates: {dup_ok}")
    print(f"    labels    : {lbl_ok}")
    for cls in CLASS_NAMES:
        ac = int(cc.get(cls, 0)); ec = EXPECTED_COUNTS[split][cls]
        print(f"    {cls}: {ac:,}  {'OK' if ac==ec else f'DIFF exp={ec:,}'}")

=== MANIFEST INTEGRITY CHECK ===
  train:
    rows      : 14,332  OK
    files     : OK
    readability (sample 50): OK
    duplicates: OK
    labels    : OK
    NV: 8,928  OK
    MEL: 3,144  OK
    BCC: 2,260  OK
  val:
    rows      : 3,012  OK
    files     : OK
    readability (sample 50): OK
    duplicates: OK
    labels    : OK
    NV: 1,886  OK
    MEL: 647  OK
    BCC: 479  OK
  test:
    rows      : 3,045  OK
    files     : OK
    readability (sample 50): OK
    duplicates: OK
    labels    : OK
    NV: 1,894  OK
    MEL: 639  OK
    BCC: 512  OK


## Section 5 - Define Transforms

Train: `Resize(256)` → `RandomCrop(224)` → flips → rotation → ColorJitter → ToTensor → Normalize  
Val/Test: `Resize(256)` → `CenterCrop(224)` → ToTensor → Normalize  
No augmented images are saved to disk.

In [7]:
train_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(degrees=30),
    T.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.02
    ),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Train transform:")
for i, step in enumerate(train_transform.transforms):
    print(f"  {i+1}. {step}")
print("\nEval transform:")
for i, step in enumerate(eval_transform.transforms):
    print(f"  {i+1}. {step}")

Train transform:
  1. Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
  2. RandomCrop(size=(224, 224), padding=None)
  3. RandomHorizontalFlip(p=0.5)
  4. RandomVerticalFlip(p=0.5)
  5. RandomRotation(degrees=[-30.0, 30.0], interpolation=nearest, expand=False, fill=0)
  6. ColorJitter(brightness=(0.85, 1.15), contrast=(0.85, 1.15), saturation=(0.85, 1.15), hue=(-0.02, 0.02))
  7. ToTensor()
  8. Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

Eval transform:
  1. Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
  2. CenterCrop(size=(224, 224))
  3. ToTensor()
  4. Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


## Section 6 - SkinLesionDataset

In [8]:
class SkinLesionDataset(Dataset):
    """
    PyTorch Dataset for the skin cancer pipeline.

    Parameters
    ----------
    df            : DataFrame — preprocessed manifest for one split.
    transform     : torchvision transform to apply to each image.
    label_mode    : 'multiclass' (NV=0, MEL=1, BCC=2) or
                    'binary'     (NV=0, MEL/BCC=1).
    include_meta  : bool — if True, __getitem__ returns (img, label, meta_dict).
                    Use a custom collate_fn when include_meta=True.
    """

    MULTICLASS = {"NV": 0, "MEL": 1, "BCC": 2}
    BINARY     = {"NV": 0, "MEL": 1, "BCC": 1}

    def __init__(self, df, transform=None,
                 label_mode="multiclass", include_meta=False):
        if label_mode not in ("multiclass", "binary"):
            raise ValueError(f"label_mode must be 'multiclass' or 'binary', got {label_mode!r}")
        self.df           = df.reset_index(drop=True)
        self.transform    = transform
        self.label_mode   = label_mode
        self.include_meta = include_meta
        self._label_map   = (self.MULTICLASS if label_mode == "multiclass"
                             else self.BINARY)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = str(row["preprocessed_full_path"])
        lbl_str = str(row["final_authoritative_label"])
        label   = self._label_map.get(lbl_str, -1)

        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)

        label_t = torch.tensor(label, dtype=torch.long)

        if not self.include_meta:
            return img, label_t

        meta = {
            "path":                    path,
            "original_full_path":      str(row.get("original_full_path", "")),
            "final_authoritative_label": lbl_str,
            "binary_label":            self.BINARY.get(lbl_str, -1),
            "canonical_match_id":      str(row.get("canonical_match_id", "")),
            "lesion_id":               str(row.get("lesion_id", "")),
        }
        return img, label_t, meta


def metadata_collate_fn(batch):
    """Collate (img, label, meta_dict) triples; meta stays as list-of-dicts."""
    imgs   = torch.stack([b[0] for b in batch])
    labels = torch.stack([b[1] for b in batch])
    metas  = [b[2] for b in batch]
    return imgs, labels, metas


def unnormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalisation for display."""
    t = tensor.clone()
    for c, (m, s) in enumerate(zip(mean, std)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)


print("SkinLesionDataset, metadata_collate_fn, unnormalize defined.")

# Smoke-test the dataset on one sample
_ds_smoke = SkinLesionDataset(dfs["train"].head(2), transform=eval_transform,
                               label_mode="multiclass", include_meta=True)
_img, _lbl, _meta = _ds_smoke[0]
print(f"  smoke test: img={_img.shape}  label={_lbl.item()}  meta_keys={list(_meta.keys())}")
del _ds_smoke, _img, _lbl, _meta

SkinLesionDataset, metadata_collate_fn, unnormalize defined.
  smoke test: img=torch.Size([3, 224, 224])  label=0  meta_keys=['path', 'original_full_path', 'final_authoritative_label', 'binary_label', 'canonical_match_id', 'lesion_id']


## Section 7 - Create DataLoaders

In [9]:
def make_loader(df, transform, label_mode, split, shuffle):
    ds = SkinLesionDataset(df, transform=transform,
                           label_mode=label_mode, include_meta=False)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

# ── Multiclass loaders ────────────────────────────────────────────────────
train_loader_multiclass = make_loader(dfs["train"], train_transform,
                                       "multiclass", "train", shuffle=True)
val_loader_multiclass   = make_loader(dfs["val"],   eval_transform,
                                       "multiclass", "val",   shuffle=False)
test_loader_multiclass  = make_loader(dfs["test"],  eval_transform,
                                       "multiclass", "test",  shuffle=False)

# ── Binary loaders ────────────────────────────────────────────────────────
train_loader_binary = make_loader(dfs["train"], train_transform,
                                   "binary", "train", shuffle=True)
val_loader_binary   = make_loader(dfs["val"],   eval_transform,
                                   "binary", "val",   shuffle=False)
test_loader_binary  = make_loader(dfs["test"],  eval_transform,
                                   "binary", "test",  shuffle=False)

print("DataLoaders created.")
for name, ldr in [
    ("train_multiclass", train_loader_multiclass),
    ("val_multiclass",   val_loader_multiclass),
    ("test_multiclass",  test_loader_multiclass),
    ("train_binary",     train_loader_binary),
    ("val_binary",       val_loader_binary),
    ("test_binary",      test_loader_binary),
]:
    n_batch = len(ldr)
    n_img   = len(ldr.dataset)
    print(f"  {name:<22}: {n_img:>6,} images  {n_batch:>4,} batches")

DataLoaders created.
  train_multiclass      : 14,332 images   448 batches
  val_multiclass        :  3,012 images    95 batches
  test_multiclass       :  3,045 images    96 batches
  train_binary          : 14,332 images   448 batches
  val_binary            :  3,012 images    95 batches
  test_binary           :  3,045 images    96 batches


## Section 8 - Validate Batches

In [10]:
print("=== BATCH VALIDATION ===")

# Store for later use in summary
batch_shapes = {}

for name, ldr, n_classes in [
    ("train multiclass", train_loader_multiclass, 3),
    ("val   multiclass", val_loader_multiclass,   3),
    ("train binary",     train_loader_binary,     2),
    ("val   binary",     val_loader_binary,       2),
]:
    imgs, labels = next(iter(ldr))
    batch_shapes[name.strip().replace(" ", "_")] = {
        "img_shape": tuple(imgs.shape),
        "lbl_shape": tuple(labels.shape),
    }

    img_min  = float(imgs.min())
    img_max  = float(imgs.max())
    lbl_vals = sorted(labels.unique().tolist())

    print(f"\n  {name}:")
    print(f"    image shape  : {tuple(imgs.shape)}")
    print(f"    image dtype  : {imgs.dtype}")
    print(f"    image min/max: {img_min:.4f} / {img_max:.4f}")
    print(f"    label shape  : {tuple(labels.shape)}")
    print(f"    label unique : {lbl_vals}  (expected {list(range(n_classes))})")
    assert imgs.shape[1:] == (3, IMG_SIZE, IMG_SIZE), \
        f"Unexpected image shape: {imgs.shape}"
    assert all(v in range(n_classes) for v in lbl_vals), \
        f"Unexpected label values: {lbl_vals}"

print("\nAll batch shape and label assertions passed.")

# Validate with metadata collate
print("\nMetadata collate smoke test:")
_ds_meta = SkinLesionDataset(dfs["train"].head(BATCH_SIZE), eval_transform,
                              label_mode="multiclass", include_meta=True)
_ldr_meta = DataLoader(_ds_meta, batch_size=4, collate_fn=metadata_collate_fn)
_imgs_m, _lbls_m, _metas_m = next(iter(_ldr_meta))
print(f"  imgs={_imgs_m.shape}  labels={_lbls_m.tolist()}")
print(f"  meta keys: {list(_metas_m[0].keys())}")
del _ds_meta, _ldr_meta, _imgs_m, _lbls_m, _metas_m

=== BATCH VALIDATION ===

  train multiclass:
    image shape  : (32, 3, 224, 224)
    image dtype  : torch.float32
    image min/max: -2.1179 / 2.6400
    label shape  : (32,)
    label unique : [0, 1, 2]  (expected [0, 1, 2])

  val   multiclass:
    image shape  : (32, 3, 224, 224)
    image dtype  : torch.float32
    image min/max: -2.0357 / 2.6400
    label shape  : (32,)
    label unique : [0]  (expected [0, 1, 2])

  train binary:
    image shape  : (32, 3, 224, 224)
    image dtype  : torch.float32
    image min/max: -2.1179 / 2.4308
    label shape  : (32,)
    label unique : [0, 1]  (expected [0, 1])

  val   binary:
    image shape  : (32, 3, 224, 224)
    image dtype  : torch.float32
    image min/max: -2.0357 / 2.6400
    label shape  : (32,)
    label unique : [0]  (expected [0, 1])

All batch shape and label assertions passed.

Metadata collate smoke test:
  imgs=torch.Size([4, 3, 224, 224])  labels=[0, 0, 0, 0]
  meta keys: ['path', 'original_full_path', 'final_authorit

## Section 9 - Batch Preview Grids

In [11]:
def save_batch_preview(loader, label_mode, save_path, title, n_cols=8):
    imgs, labels = next(iter(loader))
    n     = min(len(imgs), 32)
    imgs  = imgs[:n]
    labels = labels[:n]
    n_rows = (n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 2.2, n_rows * 2.5))
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for ax in axes_flat:
        ax.axis("off")

    if label_mode == "multiclass":
        lbl_names = CLASS_NAMES
    else:
        lbl_names = BINARY_NAMES

    for i in range(n):
        ax  = axes_flat[i]
        img = unnormalize(imgs[i]).permute(1, 2, 0).numpy()
        lbl = labels[i].item()
        ax.imshow(img)
        ax.axis("off")
        name = lbl_names[lbl] if lbl < len(lbl_names) else str(lbl)
        ax.set_title(name, fontsize=8, pad=2)

    plt.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(str(save_path), bbox_inches="tight", dpi=100)
    plt.close()
    print(f"Saved {save_path.name}")


save_batch_preview(
    train_loader_multiclass, "multiclass",
    PYTORCH_DIR / "batch_preview_train.png",
    "Train Batch - Multiclass (augmented, not saved to disk)"
)
save_batch_preview(
    val_loader_multiclass, "multiclass",
    PYTORCH_DIR / "batch_preview_val.png",
    "Val Batch - Multiclass (no augmentation)"
)

Saved batch_preview_train.png
Saved batch_preview_val.png


## Section 10 - Save Summary Files

In [12]:
# pytorch_class_distribution.csv
dist_rows = []
for split, df in dfs.items():
    cc = df["final_authoritative_label"].value_counts()
    dist_rows.append({
        "split": split,
        "NV":    int(cc.get("NV", 0)),
        "MEL":   int(cc.get("MEL", 0)),
        "BCC":   int(cc.get("BCC", 0)),
        "total": len(df),
    })
class_dist_df = pd.DataFrame(dist_rows)
class_dist_path = PYTORCH_DIR / "pytorch_class_distribution.csv"
class_dist_df.to_csv(str(class_dist_path), index=False)
print("pytorch_class_distribution.csv:")
print(class_dist_df.to_string(index=False))

pytorch_class_distribution.csv:
split   NV  MEL  BCC  total
train 8928 3144 2260  14332
  val 1886  647  479   3012
 test 1894  639  512   3045


In [13]:
# pytorch_binary_distribution.csv
bin_rows = []
for split, df in dfs.items():
    bc = df["binary_label"].value_counts()
    bin_rows.append({
        "split":       split,
        "non_cancer":  int(bc.get(0, 0)),
        "cancer_risk": int(bc.get(1, 0)),
        "total":       len(df),
    })
bin_dist_df = pd.DataFrame(bin_rows)
bin_dist_path = PYTORCH_DIR / "pytorch_binary_distribution.csv"
bin_dist_df.to_csv(str(bin_dist_path), index=False)
print("\npytorch_binary_distribution.csv:")
print(bin_dist_df.to_string(index=False))


pytorch_binary_distribution.csv:
split  non_cancer  cancer_risk  total
train        8928         5404  14332
  val        1886         1126   3012
 test        1894         1151   3045


In [14]:
# pytorch_dataset_summary.csv
summary_rows = []
for split, df in dfs.items():
    info = integrity[split]
    cc   = df["final_authoritative_label"].value_counts()
    bc   = df["binary_label"].value_counts()
    summary_rows.append({
        "split":               split,
        "row_count":           len(df),
        "expected_rows":       EXPECTED_COUNTS[split]["total"],
        "NV":                  int(cc.get("NV",  0)),
        "MEL":                 int(cc.get("MEL", 0)),
        "BCC":                 int(cc.get("BCC", 0)),
        "non_cancer":          int(bc.get(0, 0)),
        "cancer_risk":         int(bc.get(1, 0)),
        "missing_file_count":  info["n_missing"],
        "unreadable_sample":   info["n_unreadable_sample"],
        "duplicate_path_count": info["n_duplicates"],
    })
summary_df   = pd.DataFrame(summary_rows)
summary_path = PYTORCH_DIR / "pytorch_dataset_summary.csv"
summary_df.to_csv(str(summary_path), index=False)
print("pytorch_dataset_summary.csv:")
print(summary_df.to_string(index=False))

pytorch_dataset_summary.csv:
split  row_count  expected_rows   NV  MEL  BCC  non_cancer  cancer_risk  missing_file_count  unreadable_sample  duplicate_path_count
train      14332          14332 8928 3144 2260        8928         5404                   0                  0                     0
  val       3012           3012 1886  647  479        1886         1126                   0                  0                     0
 test       3045           3045 1894  639  512        1894         1151                   0                  0                     0


## Section 11 - Output File Verification

In [15]:
required_files = [
    PYTORCH_DIR / "pytorch_dataset_summary.csv",
    PYTORCH_DIR / "pytorch_class_distribution.csv",
    PYTORCH_DIR / "pytorch_binary_distribution.csv",
    PYTORCH_DIR / "batch_preview_train.png",
    PYTORCH_DIR / "batch_preview_val.png",
]
print("Output file verification:")
all_ok = True
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<42} {size:>12,} bytes")
    if not exists: all_ok = False
print("\nAll required files present." if all_ok else "\nWARNING: missing files.")

Output file verification:
  [OK] pytorch_dataset_summary.csv                         262 bytes
  [OK] pytorch_class_distribution.csv                       99 bytes
  [OK] pytorch_binary_distribution.csv                     100 bytes
  [OK] batch_preview_train.png                       1,907,521 bytes
  [OK] batch_preview_val.png                         2,041,019 bytes

All required files present.


## Section 12 - Final Summary (Copy-Paste Ready)

In [16]:
from IPython.display import display as _disp

print("=" * 68)
print("  09_pytorch_dataset_and_transforms -- FINAL SUMMARY")
print("=" * 68)

print(f"\nEnvironment:")
print(f"  torch version    : {torch.__version__}")
print(f"  CUDA available   : {CUDA_AVAILABLE}")
print(f"  GPU name         : {GPU_NAME}")
print(f"  Device selected  : {DEVICE}")

print(f"\nInput manifests:")
for split, p in manifest_paths.items():
    print(f"  {split}: {p}")

print(f"\nRow counts by split:")
for split, df in dfs.items():
    exp = EXPECTED_COUNTS[split]["total"]
    ok  = "OK" if len(df) == exp else f"DIFF exp={exp:,}"
    print(f"  {split}: {len(df):,}  {ok}")

print(f"\n3-class counts by split:")
for split, df in dfs.items():
    cc = df["final_authoritative_label"].value_counts()
    print(f"  {split}: NV={int(cc.get('NV',0)):,}  "
          f"MEL={int(cc.get('MEL',0)):,}  BCC={int(cc.get('BCC',0)):,}")

print(f"\nBinary counts by split:")
for split, df in dfs.items():
    bc = df["binary_label"].value_counts()
    print(f"  {split}: non_cancer={int(bc.get(0,0)):,}  "
          f"cancer_risk={int(bc.get(1,0)):,}")

print(f"\nMissing / unreadable / duplicate (per split):")
for split, info in integrity.items():
    print(f"  {split}: missing={info['n_missing']}  "
          f"unreadable_sample={info['n_unreadable_sample']}  "
          f"duplicates={info['n_duplicates']}")

print(f"\nBatch shapes:")
for bname, binfo in batch_shapes.items():
    print(f"  {bname:<28}: img={binfo['img_shape']}  lbl={binfo['lbl_shape']}")

print(f"\nOutput file verification:")
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<42} {size:>10,} bytes")

print(f"\nConfirmations:")
print(f"  No model training performed       : True")
print(f"  No images modified                : True")
print(f"  No augmented images saved to disk : True")
print(f"  No preprocessing repeated         : True")

print("=" * 68)

  09_pytorch_dataset_and_transforms -- FINAL SUMMARY

Environment:
  torch version    : 2.6.0+cu124
  CUDA available   : True
  GPU name         : NVIDIA GeForce RTX 4060 Ti
  Device selected  : cuda

Input manifests:
  train: C:\SKIN CANCER v2\pipe output\preprocessing\train_manifest_preprocessed.csv
  val: C:\SKIN CANCER v2\pipe output\preprocessing\val_manifest_preprocessed.csv
  test: C:\SKIN CANCER v2\pipe output\preprocessing\test_manifest_preprocessed.csv

Row counts by split:
  train: 14,332  OK
  val: 3,012  OK
  test: 3,045  OK

3-class counts by split:
  train: NV=8,928  MEL=3,144  BCC=2,260
  val: NV=1,886  MEL=647  BCC=479
  test: NV=1,894  MEL=639  BCC=512

Binary counts by split:
  train: non_cancer=8,928  cancer_risk=5,404
  val: non_cancer=1,886  cancer_risk=1,126
  test: non_cancer=1,894  cancer_risk=1,151

Missing / unreadable / duplicate (per split):
  train: missing=0  unreadable_sample=0  duplicates=0
  val: missing=0  unreadable_sample=0  duplicates=0
  test: mis

## Section 13 - Completion Summary

**Section 09 - PyTorch Dataset and Transforms is complete.**

What was accomplished:
- Preprocessed manifests loaded and integrity-checked.
- `binary_label` column derived: NV=0 (non_cancer), MEL/BCC=1 (cancer_risk).
- Train transforms defined with `RandomCrop(224)`, flips, rotation, ColorJitter, ToTensor, ImageNet normalisation.
- Eval transforms defined with `CenterCrop(224)`, ToTensor, ImageNet normalisation.
- `SkinLesionDataset` supports `label_mode='multiclass'` and `label_mode='binary'`, plus optional metadata return with `include_meta=True`.
- Six DataLoaders created (train/val/test × multiclass/binary). Train loaders shuffle; val/test do not.
- Batch shape/dtype/label assertions passed.
- Batch preview grids saved.
- Summary CSVs saved.

**What was deliberately deferred:**
- Model architecture definition
- Training loop, loss function, optimiser
- Validation/test evaluation
- Class-imbalance weighting (class weights available from distribution CSVs)

**Next notebook:** `10_training.ipynb`  
Import `SkinLesionDataset`, `train_loader_multiclass`, `val_loader_multiclass` from this file (or re-instantiate using the same manifests). Use `preprocessed_full_path` as the image column.